# TradaBoostR2

This notebook runs TradaBoostR2 as implemented in https://adapt-python.github.io/adapt/generated/adapt.utils.make_regression_da.html. 

Make sure to install adapt package (preferrably with Python 3.9), along with Tensorflow == 2.15. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from adapt.instance_based import TrAdaBoostR2, TwoStageTrAdaBoostR2
from sklearn.metrics import mean_squared_error, mean_absolute_error

import itertools

In [ ]:
seed_list = [1]
splitting_variable_list = ['CRIM'] #, 'PTRATIO', 'LSTAT']


In [ ]:
data = pd.read_csv('../datasets/boston-housing.csv')
data.columns

#data = data.dropna()
for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].astype('category').cat.codes
data

In [ ]:
#print correlations with target
target_column = 'MEDV'
correlations = data.corr()[target_column].drop(target_column)
print(correlations)
data.columns

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Reshape
from tensorflow.keras.optimizers import Adam

def get_model():
    model = Sequential()
    model.add(Dense(12, activation='relu', input_shape=(10,)))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(1e-3), loss='mean_squared_error')
    return model

In [ ]:
#ablation study for TradaBoostR2, Gaussian errors, with gaussian source domain errors
ablation_transfer_tradaboost_normal_normal = pd.DataFrame(columns = ['seed', 'splitting_variable', 'd', 'method',
                                   'n_estimators', 'lr', 'epochs', 'val_rmse', 'val_mae', 'rmse', 'mae'])

n_estimators_list = [5,20,35]
lr_list = [0.05, 0.1, 0.15]
epochs_list = [10, 25, 40]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    n_estimators_list,
    lr_list,
    epochs_list
))

for seed in seed_list:
    for splitting_variable in splitting_variable_list:
    
        #order based on this variable
        data_ = data.sort_values(by=splitting_variable)
        #remove this variable now
        data_ = data_.drop(columns = splitting_variable)
        #divide into source and target data
        data_source = data_[0:int(len(data_) / 2)]
        data_target = data_[int(len(data_) / 2):]

        predictor_columns = [col for col in data_.columns if col != target_column]

        #Split target into train, val, test

        data_target_train, data_target_temp = train_test_split(data_target, test_size = 0.602, random_state=seed)
        data_target_val, data_target_test = train_test_split(data_target_temp, test_size = 0.5, random_state=seed)
        print(len(data_target_train), len(data_target_val), len(data_target_test))

        X_source_train, y_source_train = np.array(data_source[predictor_columns], dtype=float), np.array(data_source[target_column])

        X_target_train, y_target_train = np.array(data_target_train[predictor_columns], dtype=float), np.array(data_target_train[target_column])
        X_target_val, y_target_val = np.array(data_target_val[predictor_columns], dtype=float), np.array(data_target_val[target_column])
        X_target_test, y_target_test = np.array(data_target_test[predictor_columns], dtype=float), np.array(data_target_test[target_column])
        for config in param_grid:
            n_estimators, lr, epochs = config


            method = f'TradaBoostR2'
            model = TrAdaBoostR2(get_model(),
                            n_estimators=n_estimators, lr=lr)

            model.fit(X_source_train, y_source_train, X_target_train, y_target_train, epochs = epochs, batch_size=16, verbose=0)
            preds = model.predict(X_target_test)
            val_preds = model.predict(X_target_val)
            val_rmse = np.sqrt(mean_squared_error(val_preds, y_target_val))
            val_mae = mean_absolute_error(val_preds, y_target_val)
            rmse = np.sqrt(mean_squared_error(preds, y_target_test))
            mae = mean_absolute_error(preds, y_target_test)
            ablation_transfer_tradaboost_normal_normal.loc[len(ablation_transfer_tradaboost_normal_normal)] = [seed, splitting_variable, method, n_estimators,
                                                                                                                lr, epochs, val_rmse, val_mae, rmse, mae]
            ablation_transfer_tradaboost_normal_normal.to_csv(f'results/tradaboost_ablation_housing.csv')